# 🎯 YOLOv8 Object Detection — Exploration Notebook

This notebook tests the YOLOv8 model before building the API.

Goals:
1. Load the model and check available classes
2. Run detection on a sample image
3. Understand the output format
4. Explore confidence thresholds


## 1. Imports

In [ ]:
from ultralytics import YOLO
from PIL import Image
import requests
import matplotlib.pyplot as plt
import numpy as np
print("All imports successful.")

## 2. Load the Model

In [ ]:
model = YOLO('yolov8n.pt')
print(f"Model loaded. Total classes: {len(model.names)}")
print()
for i in range(20):
    print(f"  {i}: {model.names[i]}")

## 3. All 80 Detectable Classes

In [ ]:
for idx, name in model.names.items():
    print(f"  {idx:3d}: {name}")

## 4. Run Detection on a Sample Image

In [ ]:
sample_url = "https://ultralytics.com/images/bus.jpg"
response   = requests.get(sample_url)
with open('sample.jpg', 'wb') as f:
    f.write(response.content)

image = Image.open('sample.jpg')
print(f"Image size: {image.width} x {image.height}")

plt.figure(figsize=(10, 6))
plt.imshow(image)
plt.axis('off')
plt.title('Original Image')
plt.show()

In [ ]:
results = model('sample.jpg', conf=0.25, verbose=False)
result  = results[0]

print(f"Objects detected: {len(result.boxes)}")
print()

if result.boxes is not None:
    boxes       = result.boxes.xyxy.cpu().numpy()
    confidences = result.boxes.conf.cpu().numpy()
    class_ids   = result.boxes.cls.cpu().numpy()

    for i, (box, conf, cls_id) in enumerate(zip(boxes, confidences, class_ids)):
        name = model.names[int(cls_id)]
        print(f"  {i+1}. {name:<15} conf: {conf:.4f}  bbox: [{int(box[0])}, {int(box[1])}, {int(box[2])}, {int(box[3])}]")

## 5. View Annotated Result

In [ ]:
annotated     = result.plot()
annotated_rgb = annotated[:, :, ::-1]  # BGR to RGB

plt.figure(figsize=(12, 7))
plt.imshow(annotated_rgb)
plt.axis('off')
plt.title(f'YOLOv8 Detection — {len(result.boxes)} objects found')
plt.show()

## 6. Confidence Threshold Effect

In [ ]:
thresholds = [0.10, 0.25, 0.50, 0.75]
fig, axes  = plt.subplots(1, 4, figsize=(20, 5))

for i, conf in enumerate(thresholds):
    r   = model('sample.jpg', conf=conf, verbose=False)[0]
    ann = r.plot()[:, :, ::-1]
    axes[i].imshow(ann)
    axes[i].axis('off')
    axes[i].set_title(f'Conf >= {conf:.0%}\n({len(r.boxes)} objects)')

plt.suptitle('Effect of Confidence Threshold', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## 7. Class Distribution

In [ ]:
result = model('sample.jpg', conf=0.25, verbose=False)[0]

class_counts = {}
if result.boxes is not None:
    for cls_id in result.boxes.cls.cpu().numpy():
        name = model.names[int(cls_id)]
        class_counts[name] = class_counts.get(name, 0) + 1

for cls, count in sorted(class_counts.items(), key=lambda x: -x[1]):
    print(f"  {cls:<15} {'#' * count} ({count})")

## 8. Summary

| Finding | Implication |
|---------|------------|
| 80 pre-trained classes | No training needed |
| Nano model ~6MB | Fast download, runs on CPU |
| conf=0.25 is good default | Catches most objects without too many false positives |
| result.plot() handles drawing | No need to manually draw boxes |
| BGR to RGB conversion needed | OpenCV uses BGR, PIL uses RGB |

**Next step:** Run `python api/main.py` then `streamlit run app/app.py`
